# R1 — CNN Double Descent

| | |
|---|---|
| Model | CNN5 (5-layer CNN, width multiplier k) |
| Dataset | CIFAR-10, n=5000, η=15% |
| Sweep | k ∈ {1,2,4,6,8,12,16,24,32,48,64} × 2 seeds = **22 runs** |
| Output | `fig1_r1_dd.png` — Figure 1 with interpolation threshold + param-count axis |

**Estimated time on T4: ~3–4 hours total.**

Completed runs are auto-skipped on re-run — safe to disconnect and resume.


## Step 1 — Environment

In [ ]:
!pip install -q torch torchvision numpy matplotlib seaborn tqdm pandas
import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 2 — Mount Drive + Clone repo

In [ ]:
import os, sys
from google.colab import drive

TOKEN      = 'YOUR_GITHUB_PAT_HERE'  # replace with your token   # ← replace with your PAT
REPO_DIR   = '/content/project-6699'
RESULT_DIR = '/content/drive/MyDrive/benign_overfitting/R1'

drive.mount('/content/drive')
os.makedirs(RESULT_DIR, exist_ok=True)
print(f'Results → {RESULT_DIR}')

REPO_URL = f'https://{TOKEN}@github.com/alice20030504/EECS-6699.git'
if not os.path.exists(REPO_DIR):
    os.system(f'git clone --branch yixuan {REPO_URL} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} checkout yixuan')
    os.system(f'git -C {REPO_DIR} pull')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Ready. Files:', [f for f in os.listdir('.') if f.endswith('.py')])

## Step 3 — Run R1 (22 runs, auto-skips completed ones)

In [ ]:
from run_r1 import R1_CONFIG, run_r1, plot_r1
import copy, threading, time

# Keep-alive
def _keep_alive():
    while True:
        time.sleep(60)
        try:
            from google.colab.output import eval_js; eval_js('0')
        except: pass
threading.Thread(target=_keep_alive, daemon=True).start()

cfg = copy.deepcopy(R1_CONFIG)
results = run_r1(cfg, RESULT_DIR, resume=True)
print(f'Done. {len(results)} runs.')

## Step 4 — Plot Figure 1

Generates Fig 1 with:
- Train / test error curves with seed error bands
- Interpolation threshold vertical line
- Secondary x-axis showing parameter count

**Can be re-run anytime without retraining.**

In [ ]:
from run_r1 import plot_r1
plot_r1(RESULT_DIR)

from IPython.display import Image, display
from pathlib import Path
p = Path(RESULT_DIR) / 'fig1_r1_dd.png'
if p.exists(): display(Image(str(p)))

## Step 5 — (Optional) Extend k=48, k=64 to 500 epochs

These are the widest networks; more epochs let them converge further into the benign region.

**Only deletes and reruns k=48 and k=64 (4 runs, ~1 hour). Everything else untouched.**

In [ ]:
from run_r1 import R1_CONFIG, run_r1
import copy

cfg = copy.deepcopy(R1_CONFIG)

# Extend k=48 and k=64 to 500 epochs
for extend_k in [48, 64]:
    run_r1(cfg, RESULT_DIR, resume=True, extend_k=extend_k, extend_epochs=500)

# Regenerate Figure 1 with updated results
from run_r1 import plot_r1
plot_r1(RESULT_DIR)

from IPython.display import Image, display
from pathlib import Path
p = Path(RESULT_DIR) / 'fig1_r1_dd.png'
if p.exists(): display(Image(str(p)))

## Step 6 — Summary table

In [ ]:
import pandas as pd
from src.io_utils import load_results

results = load_results(RESULT_DIR, pattern='r1_*.json')
df = pd.DataFrame([{
    'k':           r['width_multiplier'],
    'seed':        r['seed'],
    'n_params':    r['n_params'],
    'epochs':      r['epochs'],
    'train_err':   f"{r['train_error']:.3f}",
    'test_err':    f"{r['test_error']:.3f}",
    'time_min':    f"{r['wall_time_s']/60:.1f}",
} for r in results]).sort_values(['k', 'seed'])
print(df.to_string(index=False))